# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import requests
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI



In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [4]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment

In [8]:
# here is the question; type over this to ask something new
system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs in a funny way."
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [9]:
# Get gpt-4o-mini to answer, with streaming

openai = OpenAI()
def stream_brochure(question):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [10]:
stream_brochure(question)

Alright, let's unpack this magical little snippet you've got here! 🎩🐇

```python
yield from {book.get("author") for book in books if book.get("author")}
```

### What does it do?

1. **Comprehension part:**  
   `{book.get("author") for book in books if book.get("author")}`  
   - This is a **set comprehension** — it builds a set of authors.
   - For each `book` in the `books` collection (probably a list of dicts, nice guess!),
   - It picks out `book.get("author")`, but *only* if `book.get("author")` exists/truthy (so it skips missing or empty authors).
   - Because it's a **set**, duplicates are thrown out faster than soggy French fries at a fancy restaurant.

2. **yield from part:**  
   - `yield from` is like the cool, Pepe Silvia-style way to say, “Hey, take all these items from this iterable and yield them one by one.” No nested loops, no hassle.
   - In this case, it will yield each unique author individually.

### Why would you do this?

- Suppose you want to **streamline** your code by yielding authors, **one at a time**, but only unique authors who exist.
- Using `yield from` on the set means you’re handing off an iterator that will give you each unique author *without* duplicates.
- It’s a generatory goodness combo: efficient memory usage, easy to iterate over later, plus deduplication baked in.

### One tiny pitfall:

- Sets are unordered — so if you care about maintaining your book list's author order, this will scramble it like a DJ at a breakfast party.
- If order matters, consider using an OrderedDict or a list with a check.

---

So in summary:  
This code line pulls out all unique author names from a collection of books, skips books without authors, and then yields those authors one-by-one to whoever’s waiting on this generator, all neat and efficient-like.

---

Hope that made your brain peek its head out of the code jungle and say hi! 👋📚

In [11]:
# Get Llama 3.2 to answer

def get_llama_response():
    api_url = "http://localhost:11434/api/chat"
    payload = {
        "model": MODEL_LLAMA,
        "messages": [{"role": "user", "content": question}],
        "stream": True
    }
    response = requests.post(api_url, json=payload, stream=True)
    display_handle = display(Markdown(""), display_id=True)
    result = ""
    
    for line in response.iter_lines():
        if line:
            json_response = json.loads(line)
            if "message" in json_response:
                content = json_response["message"].get("content", "")
                result += content
                update_display(Markdown(result), display_id=display_handle.display_id)
            if json_response.get("done", False):
                break
    
    return result

get_llama_response()

**Code Breakdown**

The given code snippet is a part of a Python generator expression, which is used to iterate over a sequence (in this case, a list of books) and yield values based on certain conditions.

Here's a breakdown of the code:

* `yield from`: This keyword is used to delegate the execution of a sub-generator to the `yield` statements within it. It allows us to use nested generator expressions.
* `{book.get("author") for book in books if book.get("author")}`: This is the main generator expression. It's using a few concepts:
	+ `(book.get("author"))`: This is an iterator expression that yields values from the `book` dictionary, specifically the value associated with the key `"author"`. The `.get()` method is used to safely retrieve the value, avoiding a KeyError if the key doesn't exist.
	+ `for book in books`: This part of the generator expression iterates over each item (`book`) in the `books` list.
	+ `if book.get("author")`: This condition filters out any books that don't have an `"author"` key. If a book doesn't have this key, it's skipped.

**What it does**

When executed, this generator expression will:

1. Iterate over each book in the `books` list.
2. For each book, check if it has an `"author"` key.
3. If a book has an `"author"` key, yield its value (the author's name).
4. Skip any books that don't have an `"author"` key.

**Why**

This code is likely used in a context where you need to process a list of books and extract the authors' names. By using a generator expression with `yield from`, we can:

* Delegating the iteration over the books to the nested generator expression.
* Filtering out books without an `"author"` key, which avoids unnecessary processing.

This approach is efficient because it doesn't require creating an intermediate list of filtered books; instead, it yields values on-the-fly as needed.

'**Code Breakdown**\n\nThe given code snippet is a part of a Python generator expression, which is used to iterate over a sequence (in this case, a list of books) and yield values based on certain conditions.\n\nHere\'s a breakdown of the code:\n\n* `yield from`: This keyword is used to delegate the execution of a sub-generator to the `yield` statements within it. It allows us to use nested generator expressions.\n* `{book.get("author") for book in books if book.get("author")}`: This is the main generator expression. It\'s using a few concepts:\n\t+ `(book.get("author"))`: This is an iterator expression that yields values from the `book` dictionary, specifically the value associated with the key `"author"`. The `.get()` method is used to safely retrieve the value, avoiding a KeyError if the key doesn\'t exist.\n\t+ `for book in books`: This part of the generator expression iterates over each item (`book`) in the `books` list.\n\t+ `if book.get("author")`: This condition filters out any